# W10 Exercise: Reproduce a frozen experiment

Machine Learning in Engineering Science | NCKU / LAiMM | Fall 2026

**Date:** 2026-11-09. **Exercise time:** 40 minutes within the class/practice allocation, not extra time after class.

**Classroom formative practice. No new weekly graded submission is created.**

## How to work independently
1. In Colab, save a copy to your Drive and use a CPU runtime. In Jupyter, open this notebook and select a Python kernel.
2. Run the complete worked walkthrough below from top to bottom. Read every Markdown checkpoint; the walkthrough is included so no previous notebook state is needed.
3. At the Independent Exercise section, write each prediction in the response cell before running its experiment.
4. Run the starter experiment, make the requested controlled change, and record the actual result. Edit only the declared factor; preserve split rules and preprocessing.
5. Explain one mechanism and one limitation. Export the evidence JSON and discuss it with a partner.

The core exercises need no paid API, private dataset, or GPU. The agent units use clearly labeled offline simulators, not real LLM calls.

**Reproducibility boundary:** the original guided notebooks were previously executed; the new extension cells have not yet been execution-tested in this revision. Report unexpected behavior rather than claiming a check passed.


# Week 10: A complete reproducible ML workflow

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Create a frozen split

Persist sample IDs as well as a seed. A seed alone is insufficient if input row order changes.


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
data=load_diabetes(); X,y=data.data,data.target
train_ids,test_ids=train_test_split(np.arange(len(y)),test_size=.25,random_state=SEED)
print("Train/test counts:",len(train_ids),len(test_ids))


**Checkpoint.** This packaged dataset is used as a statistical teaching example, not for clinical decisions.


## 3. Fit and serialize evidence

Use a fixed hyperparameter for this exercise. A real selection exercise needs an inner validation split or cross-validation.


In [ ]:
model=make_pipeline(StandardScaler(),Ridge(alpha=20))
model.fit(X[train_ids],y[train_ids]); predictions=model.predict(X[test_ids])
summary={"test_MAE":float(mean_absolute_error(y[test_ids],predictions)),"alpha":20,
         "train_ids":train_ids.tolist(),"test_ids":test_ids.tolist()}
np.savez(ARTIFACTS/"workflow-data.npz",X=X,y=y,train_ids=train_ids,test_ids=test_ids)
print("MAE:",summary["test_MAE"])


**Checkpoint.** The test IDs and baseline configuration must be included in the report.


## 4. Reproduce in a fresh model object

Rebuild the same pipeline and fit on identical rows. Compare numerical predictions with an explicit tolerance.


In [ ]:
replica=make_pipeline(StandardScaler(),Ridge(alpha=20)).fit(X[train_ids],y[train_ids])
delta=float(np.max(np.abs(replica.predict(X[test_ids])-predictions)))
assert delta<1e-8
summary["rerun_max_delta"]=delta
print("Rerun maximum prediction difference:",delta)


**Checkpoint.** Passing this deterministic rerun is narrower than independent scientific replication.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":10,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-10-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)


## Independent Exercise: W10

Complete the three tasks below after the walkthrough. The starter cells produce observations, not your written interpretation. Fill the response fields in the next cell. Save separate records for the original run and any controlled change.

In [ ]:
exercise_id="W10"
observations={}
responses={
    'task1_prediction':'', 'task1_explanation':'',
    'task2_prediction':'', 'task2_explanation':'',
    'task3_prediction':'', 'task3_explanation':'',
    'controlled_change':'', 'limitation':'', 'ai_use_and_checks':''
}
# Fill these strings before and after each experiment. Use 'none' if no AI assistance was used.
print('Exercise record initialized:', exercise_id)

### Task 1: Preserve split membership

Freeze a train/test split of 20 row IDs. Store the IDs and check that neither overlap nor missing rows exist.

**Steps**
1. Write your prediction in `responses['task1_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task1_explanation']`; do not merely repeat a score.

**Checkpoint:** The overlap is empty and all 20 IDs are accounted for. Reordering raw rows without preserving IDs changes the experiment.

In [ ]:
frozen_order=np.random.default_rng(2026).permutation(20)
frozen_train,frozen_test=frozen_order[:15],frozen_order[15:]
observations['task1']={'train_ids':frozen_train.tolist(),'test_ids':frozen_test.tolist(),'overlap':np.intersect1d(frozen_train,frozen_test).tolist(),'all_rows_accounted_for':len(np.union1d(frozen_train,frozen_test))==20}
print(observations['task1'])

### Task 2: Refit an identical deterministic pipeline

Fit the same ridge pipeline twice on the frozen training rows. Compare test predictions using an explicit tolerance.

**Steps**
1. Write your prediction in `responses['task2_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task2_explanation']`; do not merely repeat a score.

**Checkpoint:** The deterministic rerun should agree within tolerance. This is narrower than independent scientific replication.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
ex_rng=np.random.default_rng(7); ex_X=ex_rng.normal(size=(20,3)); ex_y=2*ex_X[:,0]-ex_X[:,1]
replica_1=make_pipeline(StandardScaler(),Ridge(alpha=1)).fit(ex_X[frozen_train],ex_y[frozen_train])
replica_2=make_pipeline(StandardScaler(),Ridge(alpha=1)).fit(ex_X[frozen_train],ex_y[frozen_train])
replica_delta=float(np.max(np.abs(replica_1.predict(ex_X[frozen_test])-replica_2.predict(ex_X[frozen_test]))))
observations['task2']={'max_prediction_difference':replica_delta,'tolerance':1e-10,'within_tolerance':replica_delta<1e-10}
print(observations['task2'])

### Task 3: Write a reproducibility contract

List one item missing from a seed-only report. Save versions, split IDs, preprocessing, parameters, and a limitation in your final evidence record.

**Steps**
1. Write your prediction in `responses['task3_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task3_explanation']`; do not merely repeat a score.

**Checkpoint:** A seed alone is insufficient. Data identity, ordering, versions, and the actual configuration must also be recorded.

In [ ]:
observations['task3']={'numpy':np.__version__,'python':platform.python_version(),'split_ids':observations['task1'],'configuration':{'scaler':'training-only StandardScaler','ridge_alpha':1},'claim_scope':'same toy data and pipeline'}
print(observations['task3'])

### Finish: preserve and explain

Fill all response strings, then run the export cell. A draft file is intentionally labeled incomplete; a complete response record is not automatically a scientifically correct result. Download the JSON before the Colab session ends. Share it with a partner for explain-back, not as an additional weekly submission.

**Exit questions:** What changed? Which observation supports your explanation? What does this experiment not establish?

In [ ]:
missing_responses=[key for key,value in responses.items() if not str(value).strip()]
exercise_record={'exercise':exercise_id,'status':'draft' if missing_responses else 'ready for peer discussion','responses':responses,'observations':observations,'missing_responses':missing_responses,'environment':{'python':platform.python_version(),'numpy':np.__version__},'data_scope':'course-provided or synthetic teaching data'}
exercise_folder=Path('artifacts')/'exercises'; exercise_folder.mkdir(parents=True,exist_ok=True)
exercise_path=exercise_folder/(exercise_id+'-evidence.json')
exercise_path.write_text(json.dumps(exercise_record,indent=2,default=lambda value:value.tolist() if hasattr(value,'tolist') else str(value)))
print('Saved:',exercise_path)
print('Status:',exercise_record['status'])
print('Fields still to complete:',missing_responses)